# Min-K% & Min-K%++ Fair Comparison (standard vs attnres)

**Goal:** keep one concise pipeline for **dual-model + dual-data** comparison.

This notebook now centers on:
- full fair results (`avg_logprob`, `Min-K%`, `Min-K%++`, k=10/20/40),
- a single merged visualization for both models and both datasets,
- focused follow-up on raw Min-K and cross-k stability.


In [ ]:
# =============================================================================
# Cell 1. Environment & Imports
# =============================================================================
import sys
import json
import math
from copy import deepcopy
from pathlib import Path
from collections import defaultdict

import torch
import matplotlib.pyplot as plt
import numpy as np

# --- locate repo root ---
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / 'toygpt2').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('repo root not found (should contain toygpt2/)')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.evaluate import load_checkpoint
from toygpt2.config import DataConfig, ModelConfig
from data.data import SyntheticSequenceDataset

print('REPO_ROOT:', REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Imports OK')

## Cell 2. Method Definitions (Locked)

### Min-K% (Shi et al., 2023)
**Paper:** *Detecting Pretraining Data from Large Language Models*  
**Repo:** [swj0419/detect-pretrain-code-contamination](https://github.com/swj0419/detect-pretrain-code-contamination)

**Definition:**
Given a sequence $X = (x_1, \ldots, x_N)$ and a target model $M$:
1. Teacher-forced forward: compute $\log p_M(x_t \mid x_{<t})$ for each $t$.
2. Sort the $N$ per-token log-probabilities in ascending order.
3. Select the bottom $k\%$: $k_{\text{len}} = \lfloor N \cdot k / 100 \rfloor$ tokens with the lowest log-prob.
4. Score: $\text{Min-K\%}(X) = \frac{1}{k_{\text{len}}} \sum_{i \in \text{bottom-}k\%} \log p_M(x_i \mid x_{<i})$

Higher (less negative) score $\Rightarrow$ more likely member.

### Min-K%++ (Zhang et al., 2024)
**Paper:** *Min-K%++: Improved Baseline for Detecting Pre-Training Data from Large Language Models*  
**Repo:** [zjysteven/mink-plus-plus](https://github.com/zjysteven/mink-plus-plus)

**Definition:**
Same setup, but each token's log-prob is **z-score normalized** using the model's own
next-token distribution at that position:
1. Compute $\log p_M(v \mid x_{<t})$ and $p_M(v \mid x_{<t})$ for ALL vocab tokens $v$ at position $t$.
2. $\mu_t = \sum_{v \in V} p_M(v \mid x_{<t}) \cdot \log p_M(v \mid x_{<t})$ (probability-weighted mean = negative entropy-like quantity)
3. $\sigma_t^2 = \sum_{v \in V} p_M(v \mid x_{<t}) \cdot [\log p_M(v \mid x_{<t})]^2 - \mu_t^2$ (variance under the model distribution)
4. $z_t = \frac{\log p_M(x_t \mid x_{<t}) - \mu_t}{\sigma_t}$ (z-score normalization)
5. Score: $\text{Min-K\%++}(X) = \frac{1}{k_{\text{len}}} \sum_{i \in \text{bottom-}k\% \text{ of } z} z_i$

**Key:** $\mu_t$ and $\sigma_t$ are computed from the **probability-weighted** expectation
$\mathbb{E}_{V \sim p_M(\cdot|x_{<t})}[\log p_M(V|x_{<t})]$, NOT a uniform average over the vocabulary.
This is the correct implementation per the [official repo](https://github.com/zjysteven/mink-plus-plus).
No reference model, no calibration corpus, no extra samples needed.

### Implementation status
- **Min-K%:** Precise reproduction of the paper formula. Only simplification: we use
  synthetic data instead of natural language, and a tiny model. The scoring math is exact.
- **Min-K%++:** Precise reproduction of the z-score formula. $\mu_t$ and $\sigma_t$ are
  computed over the **full vocabulary** at each position using `probs * log_probs`
  (probability-weighted), matching the official repo. No approximation in the formula.
- **Simplifications vs paper:** (a) tiny model not real LLM, (b) synthetic data not natural
  language benchmark, (c) member/non-member defined by train-seed vs val-seed (smoke test),
  (d) small sample sizes. These are **data/scale simplifications**, not formula approximations.
- This is a **"minimal viable prototype"** -- formula-exact, data-approximate.

In [ ]:
# =============================================================================
# Cell 3. Load Checkpoint & Model
# =============================================================================
# Load BOTH checkpoints (standard + attnres) with robust path candidates.

MODEL_VARIANTS = ['standard', 'attnres']

CKPT_CANDIDATE_PATHS = {
    'standard': [
        Path('/home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/standard/ckpt_standard_last.pt'),
        Path('/home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/standard/ckpt_standard_last.pt'),
    ],
    'attnres': [
        Path('/home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/attnres/ckpt_attnres_last.pt'),
        Path('/home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/attnres/ckpt_attnres_last.pt'),
    ],
}

# Optional local fallback for standard smoke checkpoint.
CKPT_CANDIDATE_PATHS['standard'].append(
    REPO_ROOT / 'toygpt2_runs' / 'notebook_bootstrap' / 'ckpt_standard_last.pt'
)

CKPT_CANDIDATE_PATHS['attnres'].append(
    REPO_ROOT / 'toygpt2_runs' / 'notebook_bootstrap' / 'ckpt_attnres_last.pt'
)

device = torch.device('cpu')
models = {}
experiments = {}
checkpoints = {}
resolved_ckpts = {}

for variant in MODEL_VARIANTS:
    resolved = None
    tried = []
    for cand in CKPT_CANDIDATE_PATHS[variant]:
        tried.append(str(cand))
        if cand.exists():
            resolved = cand
            break

    if resolved is None:
        tried_block = ''.join(f'- {t}{chr(10)}' for t in tried)
        raise FileNotFoundError(
            'No checkpoint found for variant=' + variant + '. Tried:' + chr(10) + tried_block
        )

    model_i, exp_i, ckpt_i = load_checkpoint(resolved, device=device)
    model_i = model_i.to(device).eval()

    models[variant] = model_i
    experiments[variant] = exp_i
    checkpoints[variant] = ckpt_i
    resolved_ckpts[variant] = resolved

    mc_i = exp_i.model
    print(f'[{variant}] checkpoint: {resolved}')
    print(f'[{variant}] model_type={ckpt_i.get("model_type")} step={ckpt_i.get("step")}')
    print(f'[{variant}] n_layer={mc_i.n_layer} n_head={mc_i.n_head} n_embd={mc_i.n_embd}')
    print(f'[{variant}] vocab_size={mc_i.vocab_size} block_size={mc_i.block_size}')
    print(f'[{variant}] params={model_i.num_parameters():,}')
    print('-' * 80)

# Backward-compatible defaults for cells that expect single-model handles.
model = models['standard']
experiment = experiments['standard']
checkpoint = checkpoints['standard']
mc = experiment.model



In [ ]:
# =============================================================================
# Cell 5. Teacher-Forced Token Scoring Helper (notebook glue code)
# =============================================================================
# This helper takes input_ids and targets, runs the model, and returns
# per-token log-probabilities. No source code modification needed.
#
# Alignment note:
#   input_ids[t] -> logits[t] predicts target[t] = input_ids[t+1]
#   So logits[:, t, :] is the distribution for the NEXT token after position t.
#   We compute log_softmax(logits[:, t, :]) and gather at target[t].
#   This is the standard teacher-forced setup, matching how the model was trained.

@torch.no_grad()
def compute_per_token_logprobs(input_ids: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Compute per-token log-probability under teacher forcing.
    
    Args:
        input_ids: [B, L] token ids fed to the model
        targets:   [B, L] the ground-truth next tokens (shifted by 1)
    
    Returns:
        token_logprobs: [B, L] log p(target[t] | input_ids[:t+1]) for each t
    """
    outputs = model(input_ids.to(device))
    logits = outputs['logits']  # [B, L, V]
    log_probs = torch.log_softmax(logits, dim=-1)  # [B, L, V]
    # Gather the log-prob of the actual target token at each position
    token_logprobs = torch.gather(
        log_probs, dim=-1, index=targets.to(device).unsqueeze(-1)
    ).squeeze(-1)  # [B, L]
    return token_logprobs.cpu()


@torch.no_grad()
def compute_per_token_logprobs_and_stats(input_ids: torch.Tensor, targets: torch.Tensor):
    """
    Extended version: also returns per-position mu and sigma for Min-K%++.
    
    For Min-K%++, we need at each position t:
      mu_t    = sum_v p(v|x_{<t}) * log p(v|x_{<t})     (prob-weighted mean of log-probs)
      sigma_t = sqrt( sum_v p(v|x_{<t}) * [log p(v|x_{<t})]^2 - mu_t^2 )
    
    Returns:
        token_logprobs: [B, L]  log p(x_t | x_{<t})
        mu:             [B, L]  prob-weighted mean of log-probs over vocab
        sigma:          [B, L]  prob-weighted std of log-probs over vocab
    """
    outputs = model(input_ids.to(device))
    logits = outputs['logits']  # [B, L, V]
    
    log_probs = torch.log_softmax(logits, dim=-1)  # [B, L, V]
    probs = torch.softmax(logits, dim=-1)           # [B, L, V]
    
    # Per-token log-prob of actual target
    token_logprobs = torch.gather(
        log_probs, dim=-1, index=targets.to(device).unsqueeze(-1)
    ).squeeze(-1)  # [B, L]
    
    # Prob-weighted mean: mu_t = E_p[log p] = sum_v p(v) * log p(v)
    mu = (probs * log_probs).sum(dim=-1)  # [B, L]
    
    # Prob-weighted second moment: E_p[(log p)^2] = sum_v p(v) * (log p(v))^2
    second_moment = (probs * log_probs.pow(2)).sum(dim=-1)  # [B, L]
    
    # Variance = E[(log p)^2] - (E[log p])^2, clamped to avoid sqrt(neg)
    variance = (second_moment - mu.pow(2)).clamp(min=1e-10)
    sigma = variance.sqrt()  # [B, L]
    
    return token_logprobs.cpu(), mu.cpu(), sigma.cpu()



In [ ]:
# =============================================================================
# Cell 6. Min-K% Implementation
# =============================================================================
# Exact reproduction of Shi et al. (2023):
#   1. Get per-token logprobs (teacher-forced)
#   2. Sort ascending
#   3. Take bottom k% tokens
#   4. Average their logprobs -> Min-K% score
#
# Higher (less negative) = more likely member.

def min_k_percent_score(token_logprobs: torch.Tensor, k_percent: float) -> torch.Tensor:
    """
    Compute Min-K% score for a batch of sequences.
    
    Args:
        token_logprobs: [B, L] per-token log-probabilities
        k_percent:      float in (0, 100], percentage of lowest-logprob tokens to select
    
    Returns:
        scores: [B] Min-K% score for each sequence
    """
    B, L = token_logprobs.shape
    k_len = max(1, int(L * k_percent / 100.0))
    
    # Sort ascending, take the k_len smallest values
    sorted_logprobs, _ = torch.sort(token_logprobs, dim=-1)  # ascending
    bottom_k = sorted_logprobs[:, :k_len]  # [B, k_len]
    
    # Average
    scores = bottom_k.mean(dim=-1)  # [B]
    return scores



In [ ]:
# =============================================================================
# Cell 7. Min-K%++ Implementation (z-score normalized)
# =============================================================================
# Exact reproduction of Zhang et al. (2024):
#   1. Get per-token logprobs AND per-position mu, sigma
#   2. z_t = (logprob_t - mu_t) / sigma_t
#   3. Sort z-scores ascending, take bottom k%
#   4. Average -> Min-K%++ score
#
# Key difference from Min-K%:
#   Min-K% uses raw log-probs.
#   Min-K%++ normalizes each token's logprob by the model's own expected logprob
#   distribution at that position, making the score robust to position-dependent
#   or context-dependent variation in the model's confidence.
#
# mu_t = E_{V~p}[log p(V|x_{<t})] = sum_v p(v|x_{<t}) * log p(v|x_{<t})
# sigma_t = sqrt(Var_{V~p}[log p(V|x_{<t})])
# These are probability-weighted (not uniform) — matching the official repo.
# No reference model needed. Single forward pass.

def min_k_pp_score(
    token_logprobs: torch.Tensor,
    mu: torch.Tensor,
    sigma: torch.Tensor,
    k_percent: float,
) -> torch.Tensor:
    """
    Compute Min-K%++ score for a batch of sequences.
    
    Args:
        token_logprobs: [B, L] per-token log-probabilities
        mu:             [B, L] prob-weighted mean of log-probs at each position
        sigma:          [B, L] prob-weighted std of log-probs at each position
        k_percent:      float in (0, 100]
    
    Returns:
        scores: [B] Min-K%++ score for each sequence
    """
    B, L = token_logprobs.shape
    k_len = max(1, int(L * k_percent / 100.0))
    
    # Z-score normalization per token
    z = (token_logprobs - mu) / sigma.clamp(min=1e-8)  # [B, L]
    
    # Sort ascending, take bottom k%
    sorted_z, _ = torch.sort(z, dim=-1)
    bottom_k = sorted_z[:, :k_len]  # [B, k_len]
    
    # Average
    scores = bottom_k.mean(dim=-1)  # [B]
    return scores



In [ ]:
# =============================================================================
# Cell 9. AUC helper
# =============================================================================

def compute_auroc_manual(member_scores, nonmember_scores):
    """
    Manual AUC-ROC computation without sklearn.
    Convention: higher score = more likely member.
    """
    m_scores = np.asarray(member_scores)
    nm_scores = np.asarray(nonmember_scores)
    n_m = len(m_scores)
    n_nm = len(nm_scores)
    if n_m == 0 or n_nm == 0:
        return float('nan')

    u_count = 0
    for ms in m_scores:
        u_count += np.sum(ms > nm_scores) + 0.5 * np.sum(ms == nm_scores)
    return u_count / (n_m * n_nm)


## Pre-Registered Fair Comparison (Cross-Data)

This section pre-registers a fair protocol to test whether:
- `attnres` is naturally better for raw tail-probability detectors (Min-K%), and
- `standard` is naturally better for normalized tail-score detectors (Min-K%++).

Protocol:
- Models: `standard`, `attnres`
- Data: `repeated_pattern` and `retrieval` smoke settings (same sizes / same seed logic)
- Metrics: `avg_logprob`, `Min-K%` (k=10/20/40), `Min-K%++` (k=10/20/40)
- Reporting: all k, both models, both datasets, no cherry-picking one best point
- Scope: smoke-test only (not benchmark)


In [ ]:
# =============================================================================
# Cell 12. Fair cross-data sweep (repeated_pattern + retrieval)
# =============================================================================
K_VALUES_FAIR = [10, 20, 40]
N_MEMBER_FAIR = 64
N_NONMEMBER_FAIR = 64

DATA_PROTOCOLS = {
    'repeated_pattern': DataConfig(
        dataset_type='repeated_pattern',
        train_size=max(256, N_MEMBER_FAIR),
        val_size=max(256, N_NONMEMBER_FAIR),
        pattern_length=8,
        retrieval_pairs=4,
    ),
    'retrieval': DataConfig(
        dataset_type='retrieval',
        train_size=max(256, N_MEMBER_FAIR),
        val_size=max(256, N_NONMEMBER_FAIR),
        pattern_length=8,
        retrieval_pairs=8,
    ),
}

def build_member_nonmember_for_data(data_cfg, model_cfg, seed, n_member, n_nonmember):
    member_ds = SyntheticSequenceDataset(size=n_member, model_config=model_cfg, data_config=data_cfg, seed=seed)
    nonmember_ds = SyntheticSequenceDataset(size=n_nonmember, model_config=model_cfg, data_config=data_cfg, seed=seed + 1)
    member_in = torch.stack([member_ds[i][0] for i in range(n_member)])
    member_tg = torch.stack([member_ds[i][1] for i in range(n_member)])
    nonmember_in = torch.stack([nonmember_ds[i][0] for i in range(n_nonmember)])
    nonmember_tg = torch.stack([nonmember_ds[i][1] for i in range(n_nonmember)])
    return member_in, member_tg, nonmember_in, nonmember_tg

fair_rows = []
fair_scores = {}

for data_name, data_cfg in DATA_PROTOCOLS.items():
    member_in, member_tg, nonmember_in, nonmember_tg = build_member_nonmember_for_data(
        data_cfg=data_cfg,
        model_cfg=mc,
        seed=experiment.train.seed,
        n_member=N_MEMBER_FAIR,
        n_nonmember=N_NONMEMBER_FAIR,
    )

    for model_name in MODEL_VARIANTS:
        model = models[model_name]

        mem_lp, mem_mu, mem_sigma = compute_per_token_logprobs_and_stats(member_in, member_tg)
        nm_lp, nm_mu, nm_sigma = compute_per_token_logprobs_and_stats(nonmember_in, nonmember_tg)

        score_map = {
            'avg_logprob': {
                'member': mem_lp.mean(dim=-1).numpy(),
                'nonmember': nm_lp.mean(dim=-1).numpy(),
            }
        }

        for k in K_VALUES_FAIR:
            score_map[f'MinK_{k}'] = {
                'member': min_k_percent_score(mem_lp, k).numpy(),
                'nonmember': min_k_percent_score(nm_lp, k).numpy(),
            }
            score_map[f'MinKPP_{k}'] = {
                'member': min_k_pp_score(mem_lp, mem_mu, mem_sigma, k).numpy(),
                'nonmember': min_k_pp_score(nm_lp, nm_mu, nm_sigma, k).numpy(),
            }

        fair_scores[(data_name, model_name)] = score_map

        for metric_name, payload in score_map.items():
            auc = compute_auroc_manual(payload['member'], payload['nonmember'])
            if metric_name == 'avg_logprob':
                metric_family = 'avg_logprob'
                k_val = None
            elif metric_name.startswith('MinKPP_'):
                metric_family = 'MinKPP'
                k_val = int(metric_name.split('_')[1])
            elif metric_name.startswith('MinK_'):
                metric_family = 'MinK'
                k_val = int(metric_name.split('_')[1])
            else:
                metric_family = metric_name
                k_val = None

            fair_rows.append({
                'model': model_name,
                'data': data_name,
                'metric': metric_family,
                'k': k_val,
                'auc': float(auc),
            })

for row in fair_rows:
    other = 'attnres' if row['model'] == 'standard' else 'standard'
    competitor_auc = None
    for r in fair_rows:
        if r['model'] == other and r['data'] == row['data'] and r['metric'] == row['metric'] and r['k'] == row['k']:
            competitor_auc = r['auc']
            break
    row['above_random'] = bool(row['auc'] > 0.5)
    row['above_other_model'] = (None if competitor_auc is None else bool(row['auc'] > competitor_auc))

try:
    import pandas as pd
    fair_df = pd.DataFrame(fair_rows).sort_values(['data', 'metric', 'k', 'model'], na_position='first').reset_index(drop=True)
    print('Fair comparison table (all metrics, all k, both models, both datasets):')
    print(fair_df.to_string(index=False))
except Exception:
    fair_df = None
    print('Fair comparison rows:')
    for row in fair_rows:
        print(row)


In [ ]:
# =============================================================================
# Cell 13. Fair combined visualization (Min-K only; cross-data, cross-model)
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(17, 11))
fig.suptitle('Fair Standard vs AttnRes (Min-K only; repeated_pattern + retrieval)', fontsize=14)

# Panel A: Min-K AUC vs k (all model/data curves)
ax = axes[0, 0]
for data_name in DATA_PROTOCOLS:
    for model_name in MODEL_VARIANTS:
        y = []
        for k in K_VALUES_FAIR:
            row = next(
                r for r in fair_rows
                if r['data'] == data_name and r['model'] == model_name and r['metric'] == 'MinK' and r['k'] == k
            )
            y.append(row['auc'])
        style = '-' if data_name == 'repeated_pattern' else '--'
        ax.plot(K_VALUES_FAIR, y, marker='o', linestyle=style, linewidth=2, label=f'{model_name} | {data_name}')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.set_title('AUC vs k (Min-K%)')
ax.set_xlabel('k (%)')
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.0, 1.0)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel B: Delta (attnres - standard) across k, split by dataset
ax = axes[0, 1]
for data_name in DATA_PROTOCOLS:
    deltas = []
    for k in K_VALUES_FAIR:
        a = next(r['auc'] for r in fair_rows if r['data'] == data_name and r['model'] == 'attnres' and r['metric'] == 'MinK' and r['k'] == k)
        s = next(r['auc'] for r in fair_rows if r['data'] == data_name and r['model'] == 'standard' and r['metric'] == 'MinK' and r['k'] == k)
        deltas.append(float(a - s))
    style = '-' if data_name == 'repeated_pattern' else '--'
    ax.plot(K_VALUES_FAIR, deltas, marker='o', linestyle=style, linewidth=2, label=f'{data_name}: attnres-standard')
ax.axhline(0.0, color='gray', linestyle=':', linewidth=1)
ax.set_title('AUC delta vs k (Min-K%, attnres - standard)')
ax.set_xlabel('k (%)')
ax.set_ylabel('Delta AUC')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel C: Representative Min-K score distribution (retrieval, k=20)
ax = axes[1, 0]
rep_data = 'retrieval'
rep_metric_key = 'MinK_20'
for model_name, color in [('standard', 'tab:blue'), ('attnres', 'tab:orange')]:
    m = fair_scores[(rep_data, model_name)][rep_metric_key]['member']
    nm = fair_scores[(rep_data, model_name)][rep_metric_key]['nonmember']
    ax.hist(m, bins=24, histtype='step', linewidth=2, color=color, label=f'{model_name} member')
    ax.hist(nm, bins=24, histtype='step', linewidth=2, linestyle='--', color=color, label=f'{model_name} non-member')
ax.set_title('Representative distribution (retrieval, Min-K%, k=20)')
ax.set_xlabel('score')
ax.set_ylabel('count')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel D: Mean Min-K AUC by k (averaged over repeated_pattern + retrieval)
ax = axes[1, 1]
x = np.arange(len(K_VALUES_FAIR))
w = 0.35
vals_std = []
vals_att = []
for k in K_VALUES_FAIR:
    auc_std = [r['auc'] for r in fair_rows if r['model'] == 'standard' and r['metric'] == 'MinK' and r['k'] == k]
    auc_att = [r['auc'] for r in fair_rows if r['model'] == 'attnres' and r['metric'] == 'MinK' and r['k'] == k]
    vals_std.append(float(np.mean(auc_std)))
    vals_att.append(float(np.mean(auc_att)))
ax.bar(x - w / 2, vals_std, width=w, label='standard')
ax.bar(x + w / 2, vals_att, width=w, label='attnres')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([f'MinK@{k}' for k in K_VALUES_FAIR])
ax.set_ylim(0.0, 1.0)
ax.set_title('Mean AUC summary (Min-K only)')
ax.set_ylabel('AUC-ROC')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()



## Focus of this notebook: raw Min-K% and cross-k stability

This notebook keeps **all full results** (avg_logprob / Min-K% / Min-K%++, k=10/20/40, standard+attnres, repeated_pattern+retrieval).

Re-organization for this iteration:
- **Part A (full results):** Cells 12-13 remain the complete reporting section.
- **Part B (raw Min-K%):** dedicated model-vs-model summary on raw tail-probability scores.
- **Part C (cross-k stability):** dedicated stability diagnostics across k for Min-K and Min-K++.

Reason: in the current smoke setup, attnres' main advantage appears in raw Min-K and potential cross-k stability, while standard can still lead on Min-K++ peaks.


In [ ]:
# =============================================================================
# Cell 17. Raw Min-K% summary (standard vs attnres)
# =============================================================================

def _lookup_auc(rows, *, data, model, metric, k):
    for r in rows:
        if r['data'] == data and r['model'] == model and r['metric'] == metric and r['k'] == k:
            return float(r['auc'])
    raise KeyError(f'missing row: data={data}, model={model}, metric={metric}, k={k}')

DATASETS_ORDER = list(DATA_PROTOCOLS.keys())
raw_mink_summary_rows = []

for k in K_VALUES_FAIR:
    std_by_data = [_lookup_auc(fair_rows, data=d, model='standard', metric='MinK', k=k) for d in DATASETS_ORDER]
    att_by_data = [_lookup_auc(fair_rows, data=d, model='attnres', metric='MinK', k=k) for d in DATASETS_ORDER]

    std_auc = float(np.mean(std_by_data))
    att_auc = float(np.mean(att_by_data))
    deltas = [a - s for a, s in zip(att_by_data, std_by_data)]

    raw_mink_summary_rows.append({
        'k': int(k),
        'standard_auc': std_auc,
        'attnres_auc': att_auc,
        'delta_auc': float(att_auc - std_auc),
        'attnres_above_standard_all_datasets': bool(all(d > 0 for d in deltas)),
    })

# Required table: k / standard_auc / attnres_auc / delta_auc
try:
    import pandas as pd
    raw_mink_summary_df = pd.DataFrame(raw_mink_summary_rows)
    raw_mink_summary_df = raw_mink_summary_df[['k', 'standard_auc', 'attnres_auc', 'delta_auc', 'attnres_above_standard_all_datasets']]
    print('Raw Min-K% comparison table (AUC averaged over repeated_pattern + retrieval):')
    print(raw_mink_summary_df.to_string(index=False))
except Exception:
    raw_mink_summary_df = None
    print('Raw Min-K% comparison table (fallback print):')
    for row in raw_mink_summary_rows:
        print(row)

# Auto conclusion text for this section
all_k_attn_better = all(r['delta_auc'] > 0 for r in raw_mink_summary_rows)
max_gap_row = max(raw_mink_summary_rows, key=lambda x: x['delta_auc'])
all_data_consistent = all(r['attnres_above_standard_all_datasets'] for r in raw_mink_summary_rows)

raw_mink_claim = {
    'all_k_attnres_gt_standard': bool(all_k_attn_better),
    'largest_gap_k': int(max_gap_row['k']),
    'largest_gap_delta_auc': float(max_gap_row['delta_auc']),
    'consistent_per_dataset_across_k': bool(all_data_consistent),
}

print()
print('Raw Min-K% conclusion (auto):')
print(f"- attnres > standard on all k (mean across datasets): {raw_mink_claim['all_k_attnres_gt_standard']}")
print(f"- largest gap at k={raw_mink_claim['largest_gap_k']} with delta={raw_mink_claim['largest_gap_delta_auc']:.4f}")
print(f"- attnres > standard in every dataset for every k: {raw_mink_claim['consistent_per_dataset_across_k']}")
if raw_mink_claim['all_k_attnres_gt_standard']:
    print('- This run supports attnres as a stronger raw tail-probability detector (smoke-test scope).')
else:
    print('- This run does not show uniform raw Min-K superiority; interpret as mixed/local advantages.')


In [ ]:
# =============================================================================
# Cell 18. Raw Min-K% line plot
# =============================================================================
k_vals = [r['k'] for r in raw_mink_summary_rows]
std_vals = [r['standard_auc'] for r in raw_mink_summary_rows]
att_vals = [r['attnres_auc'] for r in raw_mink_summary_rows]
delta_vals = [r['delta_auc'] for r in raw_mink_summary_rows]

plt.figure(figsize=(8, 5))
plt.plot(k_vals, std_vals, marker='o', linewidth=2, label='standard')
plt.plot(k_vals, att_vals, marker='o', linewidth=2, label='attnres')
for k, d in zip(k_vals, delta_vals):
    plt.text(k, max(std_vals + att_vals) + 0.005, f'Δ={d:+.3f}', ha='center', va='bottom', fontsize=9)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.ylim(0.0, 1.0)
plt.xlabel('k (%)')
plt.ylabel('AUC-ROC')
plt.title('Raw Min-K% comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# =============================================================================
# Cell 19. Cross-k stability summary (Min-K and Min-K++)
# =============================================================================

def compute_cross_k_stability(rows, metric_type):
    out = []
    for model_name in MODEL_VARIANTS:
        auc_by_k = []
        for k in K_VALUES_FAIR:
            vals = [
                float(r['auc'])
                for r in rows
                if r['model'] == model_name and r['metric'] == metric_type and r['k'] == k
            ]
            auc_by_k.append(float(np.mean(vals)))

        range_across_k = float(max(auc_by_k) - min(auc_by_k))
        std_across_k = float(np.std(auc_by_k))
        drop_10_to_40 = float(abs(auc_by_k[0] - auc_by_k[-1]))
        peak_idx = int(np.argmax(auc_by_k))
        peak_auc = float(auc_by_k[peak_idx])
        peak_k = int(K_VALUES_FAIR[peak_idx])

        out.append({
            'model': model_name,
            'metric_type': 'Min-K' if metric_type == 'MinK' else 'Min-K++',
            'range_across_k': range_across_k,
            'std_across_k': std_across_k,
            'drop_10_to_40': drop_10_to_40,
            'peak_auc': peak_auc,
            'peak_k': peak_k,
        })
    return out

stability_rows = []
stability_rows.extend(compute_cross_k_stability(fair_rows, 'MinK'))
stability_rows.extend(compute_cross_k_stability(fair_rows, 'MinKPP'))

try:
    import pandas as pd
    stability_df = pd.DataFrame(stability_rows)
    stability_df = stability_df[['model', 'metric_type', 'range_across_k', 'std_across_k', 'drop_10_to_40', 'peak_auc', 'peak_k']]
    print('Cross-k stability summary table (AUC averaged over repeated_pattern + retrieval):')
    print(stability_df.to_string(index=False))
except Exception:
    stability_df = None
    print('Cross-k stability summary table (fallback print):')
    for row in stability_rows:
        print(row)

# Compact diagnostic statements
for metric_type in ['Min-K', 'Min-K++']:
    sub = [r for r in stability_rows if r['metric_type'] == metric_type]
    more_stable = min(sub, key=lambda r: (r['std_across_k'], r['range_across_k'], r['drop_10_to_40']))['model']
    higher_peak = max(sub, key=lambda r: r['peak_auc'])['model']
    print(f"\n[{metric_type}] more stable across k: {more_stable}")
    print(f"[{metric_type}] higher peak AUC across k: {higher_peak}")



In [ ]:
# =============================================================================
# Cell 20. Cross-k stability visualization (Min-K only)
# =============================================================================

min_k_rows = [r for r in stability_rows if r['metric_type'] == 'Min-K']

def _mk_stat(model_name, key):
    row = next(r for r in min_k_rows if r['model'] == model_name)
    return float(row[key])

stats = ['range_across_k', 'std_across_k', 'drop_10_to_40', 'peak_auc']
labels = ['range across k (low=stable)', 'std across k (low=stable)', 'drop 10->40 (low=stable)', 'peak AUC (high=better)']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Cross-k stability & peak (Min-K only, standard vs attnres)', fontsize=13)

for ax, key, title in zip(axes.flatten(), stats, labels):
    vals = [_mk_stat('standard', key), _mk_stat('attnres', key)]
    ax.bar(['standard', 'attnres'], vals, color=['tab:blue', 'tab:orange'])
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.show()



## Final interpretation (balanced, smoke-test scope)

- **attnres favorable standards:** prioritize the Raw Min-K% section (Cells 17-18) and the Min-K stability rows in Cell 19.
- **standard favorable standard:** keep reporting Min-K++ peaks (Cell 13 + Cell 19 `peak_auc`).
- **Balanced reading:** do not collapse this into a single winner. The fair claim is metric-family specific: raw tail-based metrics can favor attnres, while normalized tail-score metrics can favor standard.
- **Scope warning:** these are smoke-test observations, not benchmark-level proof.
- **Next validation:** keep the same protocol and extend to larger retrieval settings and TinyStories subsets before strong claims.


## Concise takeaway (smoke-test scope)

- Keep judgment on the merged dual-model/dual-data plots and tables above; do not cherry-pick one k.
- attnres should be discussed via raw Min-K level + cross-k stability evidence.
- standard should still be discussed where Min-K++ reaches higher peaks.

These are smoke-test observations, not benchmark-level claims.
